
# Task 1 — Data Audit & Exploration

**Project:** GrowMar Employee Attrition ML

**Objective:** Audit the employee dataset for structural issues, missing values, outliers, class imbalance, and data leakage risks.

**Tools:** Python, pandas, NumPy, Matplotlib, Seaborn, JupyterLab, Docker.

**Target variable:** `left_within_12m`

**Purpose:** Produce a reproducible audit notebook and a clear report documenting findings and decisions before building machine learning models.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path("/workspace")
if not (PROJECT_ROOT / "data").is_dir():
    raise FileNotFoundError("Expected project data directory at /workspace/data")

DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "phase-1" / "reports"
EVIDENCE_DIR = PROJECT_ROOT / "phase-1" / "evidence"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready!")
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Report directory:", REPORT_DIR)
print("Evidence directory:", EVIDENCE_DIR)


Environment ready!
Project root: /workspace
Data directory: /workspace/data
Report directory: /workspace/phase-1/reports
Evidence directory: /workspace/phase-1/evidence


In [2]:
print("Data directory exists:", DATA_DIR.exists())
print("Dataset exists:", (DATA_DIR / "attrition_dataset.csv").exists())

Data directory exists: True
Dataset exists: True


In [3]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
print("Workspace exists:", Path("/workspace").exists())
print("CSV exists:", Path("/workspace/data/attrition_dataset.csv").exists())

Current working directory: /workspace/phase-1/notebooks
Workspace exists: True
CSV exists: True


In [4]:
df = pd.read_csv(DATA_DIR / "attrition_dataset.csv")

NameError: name 'pd' is not defined


## 1. Dataset Overview

The purpose of this section is to understand the size and structure of the employee dataset before performing further analysis.


In [ ]:

# Dataset overview

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nFirst five records:")
display(df.head())



## 2. Column Summary

This section examines the data type, missing values, and distinct values in each column. It helps identify fields that require further investigation.


In [ ]:

# Column summary

column_summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2),
    "Unique Values": df.nunique()
})

display(column_summary)


In [ ]:

# Check exact duplicate rows

duplicate_count = df.duplicated().sum()

print("Number of exact duplicate rows:", duplicate_count)


In [ ]:
# Check the distribution of the target column
print("Target column distribution:")
print(df["left_within_12m"].value_counts())

print("\nTarget distribution in percentage:")
print((df["left_within_12m"].value_counts(normalize=True) * 100).round(2))

# Visualize the target distribution
sns.countplot(data=df, x="left_within_12m")
plt.title("Distribution of Employee Attrition")
plt.xlabel("Left Within 12 Months (0 = No, 1 = Yes)")
plt.ylabel("Number of Records")
plt.savefig(
    REPORT_DIR / "attrition_distribution.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

## 3. Missing Value Analysis

This section identifies missing values in the dataset and calculates their percentage for each column. The results help determine which columns require further investigation and how missing data should be handled during preprocessing.

In [ ]:
# Missing value analysis
missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

# Display columns containing missing values
missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
]

# Display the 10 columns with the most missing values
display(
    missing_summary.sort_values(
        by="Missing Percentage",
        ascending=False
    ).head(10)
)

## 4. Duplicate Record Analysis

This section checks for exact duplicate rows in the dataset. Duplicate records may affect statistical analysis and model performance, so they should be investigated before deciding whether any records need to be removed.

In [ ]:
# Check for exact duplicate rows
duplicate_count = df.duplicated().sum()

print("Number of exact duplicate rows:", duplicate_count)

## 5. Outlier Detection

This section identifies unusual values in numerical features that may indicate extreme employee characteristics or potential data quality issues. The Interquartile Range (IQR) method and Z-score method will be used to detect potential outliers. These values will be investigated before deciding whether any action is necessary.

In [ ]:
# Identify numerical columns
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

print("Number of numerical columns:", len(numeric_columns))
print("\nNumerical columns:")

for i, column in enumerate(numeric_columns, start=1):
    print(f"{i}. {column}")

### 5.1 Selecting Numerical Features

Not all numerical columns are suitable for outlier detection. Identifier columns and target variables are excluded because they do not represent continuous measurements. We will focus on employee-related numerical features such as salary, tenure, working hours, and attendance.

In [ ]:
# Select the required columns
outlier_columns = [
    "annual_fixed_pay",
    "avg_hours_worked_3m",
    "tenure_months"
]

# Detect outliers using the IQR method
for column in outlier_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print(f"\nColumn: {column}")
    print("Lower bound:", round(lower_bound, 2))
    print("Upper bound:", round(upper_bound, 2))
    print("Number of potential outliers:", len(outliers))

### 5.2 Outlier Detection Using the Z-score Method

The Z-score method measures how far a value is from the mean in terms of standard deviations. Values with an absolute Z-score greater than 3 are flagged as potential outliers. The results will be compared with the IQR method to understand how the two approaches identify unusual values.

In [ ]:
# Detect outliers using the Z-score method
from scipy import stats
for column in outlier_columns:
    values = df[column].dropna()

    z_scores = stats.zscore(values)

    outlier_count = (abs(z_scores) > 3).sum()

    print(f"\nColumn: {column}")
    print("Number of potential outliers:", outlier_count)

### 5.3 Comparison of IQR and Z-score Methods

This section compares the number of potential outliers detected by the IQR and Z-score methods for salary, average working hours, and employee tenure. Differences in the results help identify how each method responds to the distribution of the data.

In [ ]:
# Compare IQR and Z-score results
comparison = []

for column in outlier_columns:
    values = df[column].dropna()

    # IQR method
    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    iqr_count = (
        (values < lower_bound) |
        (values > upper_bound)
    ).sum()

    # Z-score method
    z_scores = stats.zscore(values)
    zscore_count = (abs(z_scores) > 3).sum()

    # Store results
    comparison.append({
        "Feature": column,
        "IQR Outliers": int(iqr_count),
        "Z-score Outliers": int(zscore_count)
    })

comparison_df = pd.DataFrame(comparison)
display(comparison_df)

## 6. Data Leakage Analysis

Data leakage occurs when information that would not be available at prediction time is used as a model feature. This can lead to misleading model performance. In this section, we will review columns that may reveal future employee attrition and identify fields that should be excluded from model training.

In [ ]:
# Display all dataset columns for leakage review
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

In [ ]:
# Columns requiring data leakage review
review_columns = [
    "row_id",
    "employee_id",
    "observation_date",
    "left_within_12m",
    "voluntary_left_within_12m",
    "exit_date",
    "resignation_date",
    "exit_reason",
    "exit_interview_done",
    "fnf_settlement_amount"
]

# Display their data types and missing values
leakage_review = pd.DataFrame({
    "Data Type": df[review_columns].dtypes,
    "Missing Values": df[review_columns].isnull().sum(),
    "Unique Values": df[review_columns].nunique()
})

display(leakage_review)

### 6.2 Data Leakage Review and Decisions

The review identifies several columns that require special handling before model training. The target columns will be excluded from the input features. Exit-related fields will also be excluded if they contain information recorded after the prediction date. The row and employee identifiers will not be used as predictive features. The observation date will be reviewed to ensure that the dataset follows the intended prediction timeline.

In [ ]:
# Document initial data leakage decisions
leakage_decisions = pd.DataFrame({
    "Column": [
        "row_id",
        "employee_id",
        "observation_date",
        "left_within_12m",
        "voluntary_left_within_12m",
        "exit_date",
        "resignation_date",
        "exit_reason",
        "exit_interview_done",
        "fnf_settlement_amount"
    ],
    "Decision": [
        "Exclude",
        "Exclude",
        "Review date and prediction timeline",
        "Target - not an input feature",
        "Outcome label - not an input feature",
        "Exclude if recorded after prediction date",
        "Exclude if it reveals future resignation",
        "Exclude from model inputs",
        "Exclude if recorded after employee exit",
        "Exclude if generated after employee exit"
    ]
})

display(leakage_decisions)

In [ ]:

# Check duplicate rows across all columns
print("Dataset shape:", df.shape)
print("Exact duplicate rows:", df.duplicated().sum())

# Check duplicate row IDs
print("Duplicate row IDs:", df["row_id"].duplicated().sum())

# Check duplicate employee IDs
print("Duplicate employee IDs:", df["employee_id"].duplicated().sum())

# Check duplicates after excluding row_id
df_without_row_id = df.drop(columns=["row_id"])
print(
    "Duplicate rows excluding row_id:",
    df_without_row_id.duplicated().sum()
)

# Display the last 10 records for inspection
display(df.tail(10))


In [ ]:

# Verify the six duplicate records listed in the answer key
duplicate_ids = [
    "EMP-0325",
    "EMP-0238",
    "EMP-0483",
    "EMP-0060",
    "EMP-0237",
    "EMP-0570"
]

duplicate_records = df[df["employee_id"].isin(duplicate_ids)].copy()

display(
    duplicate_records[
        ["row_id", "employee_id", "observation_date"]
    ].sort_values("employee_id")
)

print("Matching records found:", len(duplicate_records))
print(
    "Duplicates excluding row_id:",
    df.drop(columns=["row_id"]).duplicated().sum()
)


In [ ]:

# Verify missing values in the two answer-key columns
columns_to_check = ["commute_distance_km", "highest_education"]

for column in columns_to_check:
    print(f"\n{column}")
    print("Total missing values:", df[column].isna().sum())
    display(
        df.loc[df[column].isna(), ["row_id", "employee_id", column]]
    )


In [ ]:

# Inspect work_location categories
print("Raw work_location values:")
print(sorted(df["work_location"].dropna().unique()))

print("\nCounts by raw work_location value:")
display(df["work_location"].value_counts(dropna=False).to_frame("count"))


In [ ]:

# Verify the three planted salary errors
salary_ids = ["EMP-0034", "EMP-0601", "EMP-0433"]

salary_check = df.loc[
    df["employee_id"].isin(salary_ids),
    ["employee_id", "annual_fixed_pay", "comp_ratio"]
]

display(salary_check.sort_values("employee_id"))


### Answer-Key Verification: Salary Errors

The answer key identifies three planted salary errors in `annual_fixed_pay`. The affected records are EMP-0034, EMP-0433 and EMP-0601. Their recorded annual fixed pay values are 4,888,200, 18,671,760 and 3,606,000, respectively. The answer key indicates that these values are ten times too high and that the corresponding `comp_ratio` values no longer match.

These records are flagged for correction during the cleaning stage. The original dataset remains unchanged during the audit.

### Answer-Key Comparison: Other Planted Issues

The `answer_key_planted_dirt` sheet lists 98 planted issues:

- **70 missing values:** 55 missing `commute_distance_km` values and 15 missing `highest_education` values.
- **13 inconsistent categories:** variations in `work_location`, including differences in capitalization, spelling and alternative city names.
- **6 duplicate rows:** the answer key identifies six appended duplicate records. The independent audit found six duplicate records after excluding `row_id`, but zero exact duplicates across all columns. This discrepancy is documented for further verification.
- **4 negative tenure values:** negative `tenure_months` values caused by observation dates preceding join dates.
- **3 salary errors:** `annual_fixed_pay` values ten times too high for EMP-0034, EMP-0601 and EMP-0433.
- **2 impossible values:** `attendance_rate_3m` values above 1.0.

The independent audit detected the major categories of planted problems. However, category-level checks alone do not confirm every planted record. The missing-value and category entries should be verified against the source dataset before the audit is considered fully reconciled.

In [ ]:
# Inspect repeated employee records and their observation dates

repeated_employees = df[
    df["employee_id"].duplicated(keep=False)
].sort_values("employee_id")

display(
    repeated_employees[
        ["employee_id", "observation_date", "left_within_12m"]
    ].sort_values(
        ["employee_id", "observation_date"]
    )
)

## 7. Attrition Label Analysis

This section examines the distribution of the target variable, `left_within_12m`, to determine whether the dataset contains an imbalance between employees who left and those who stayed. Understanding the class distribution helps guide the evaluation metrics and modelling approach used in subsequent tasks.

In [ ]:
# Count attrition labels
attrition_counts = df["left_within_12m"].value_counts().sort_index()

# Calculate percentages
attrition_percentages = (
    df["left_within_12m"].value_counts(normalize=True).sort_index() * 100
).round(2)

# Display results
attrition_summary = pd.DataFrame({
    "Employee Records": attrition_counts,
    "Percentage": attrition_percentages
})

display(attrition_summary)

## 8. Data Dictionary

This section documents the meaning and purpose of important dataset columns. It helps explain the employee attributes, attendance and performance measures, and attrition labels used during the audit. The data dictionary also supports decisions about missing values, feature selection, and data leakage prevention.

In [ ]:
# Load the official data dictionary
dictionary_df = pd.read_csv(DATA_DIR / "data_dictionary.csv")

# Select definitions for the attrition dataset only
attrition_dictionary = dictionary_df[
    dictionary_df["sheet"] == "attrition_dataset"
]

# Display the data dictionary
display(
    attrition_dictionary[
        ["column", "data_type", "description",
         "null_count", "null_percent", "distinct_values", "example"]
    ]
)

## 9. Data Audit Visualizations

This section presents visual summaries of the attrition distribution and missing values in the dataset. These visualizations help communicate the audit findings clearly and identify data-quality considerations before model development.

In [ ]:
# Visualize the attrition label distribution
sns.countplot(data=df, x="left_within_12m")

plt.title("Employee Attrition Distribution")
plt.xlabel("Left Within 12 Months (0 = No, 1 = Yes)")
plt.ylabel("Number of Records")
plt.xticks([0, 1], ["Stayed (0)", "Left (1)"])

plt.tight_layout()
plt.show()

In [ ]:
# Select columns that contain missing values
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(
    ascending=False
)

# Plot the 10 columns with the most missing values
if not missing_counts.empty:
    missing_counts.head(10).plot(kind="bar")

    plt.title("Top 10 Columns with Missing Values")
    plt.xlabel("Column")
    plt.ylabel("Number of Missing Values")
    plt.xticks(rotation=45, ha="right")

    plt.tight_layout()
    plt.savefig(
    REPORT_DIR / "missing_values_top10.png",
    dpi=150,
    bbox_inches="tight"
)
    plt.show()
else:
    print("No missing values found.")

## 10. Audit Findings and Data-Quality Decisions

This section summarizes the key findings from the dataset audit. The findings cover missing values, duplicate records, potential outliers, attrition class imbalance, and possible data leakage. These observations will guide preprocessing and feature selection in subsequent machine learning tasks.

In [ ]:
# Summarize key dataset audit findings
audit_findings = pd.DataFrame({
    "Audit Check": [
        "Dataset Size",
        "Exact Duplicate Rows",
        "Repeated Employee IDs",
        "Attrition Label 0",
        "Attrition Label 1"
    ],
    "Finding": [
        f"{df.shape[0]} rows and {df.shape[1]} columns",
        str(df.duplicated().sum()),
        str(df["employee_id"].duplicated().sum()),
        str((df["left_within_12m"] == 0).sum()),
        str((df["left_within_12m"] == 1).sum())
    ]
})

display(audit_findings)

### 10.1 Data-Quality Decisions

The audit identified several areas that require attention before model development. Missing values will be handled according to the meaning of each feature. Repeated employee IDs will be investigated to distinguish legitimate historical records from data-quality issues. Potential outliers will be reviewed using the IQR and Z-score methods. The attrition target is imbalanced, with fewer employees leaving than staying. Target labels, identifiers, and information unavailable at the prediction date will be excluded from model inputs to prevent data leakage.

In [ ]:
# Display the IQR and Z-score comparison
display(comparison_df)

In [ ]:
%pip install tabulate
# Prepare the audit report

In [ ]:

# Generate the complete Task 1 audit report
from pathlib import Path

report_path = REPORT_DIR / "data-audit.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

# Missing-value summary
missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": df.isnull().sum().values,
    "Missing Percentage": (df.isnull().mean().values * 100).round(2)
})
missing_report = missing_report[
    missing_report["Missing Count"] > 0
].sort_values("Missing Count", ascending=False)

# Check repeated employee IDs
duplicate_without_row_id = df.drop(columns=["row_id"]).duplicated().sum()

# Check category values for whitespace/capitalization differences.
# This identifies candidates for review; it does not automatically change data.
category_columns = df.select_dtypes(include=["object", "string"]).columns
category_issues = []

for column in category_columns:
    values = df[column].dropna().astype(str)
    normalized = values.str.strip().str.casefold()

    if values.nunique() > normalized.nunique():
        category_issues.append({
            "Column": column,
            "Raw unique values": values.nunique(),
            "Normalized unique values": normalized.nunique()
        })

category_report = pd.DataFrame(
    category_issues,
    columns=["Column", "Raw unique values", "Normalized unique values"]
)

# Check known bounded measurements and non-negative numeric fields.
invalid_checks = []

bounded_ranges = {
    "attendance_rate_3m": (0, 1),
    "wfh_share_3m": (0, 1),
    "last_performance_rating": (1, 5),
    "last_engagement_score": (1, 5)
}

for column, (lower, upper) in bounded_ranges.items():
    if column in df.columns:
        invalid = df[column].notna() & (
            (df[column] < lower) | (df[column] > upper)
        )
        invalid_checks.append({
            "Column": column,
            "Validation rule": f"{lower} to {upper}",
            "Invalid non-missing values": int(invalid.sum())
        })

for column in [
    "tenure_months",
    "total_experience_years",
    "previous_employers_count",
    "commute_distance_km",
    "annual_fixed_pay",
    "raises_received",
    "promotions_received",
    "late_arrivals_3m",
    "avg_overtime_hours_per_month_3m",
    "avg_hours_worked_3m",
    "unpaid_leave_days_6m",
    "paid_leave_days_6m"
]:
    if column in df.columns:
        invalid = df[column].notna() & (df[column] < 0)
        invalid_checks.append({
            "Column": column,
            "Validation rule": "Non-negative",
            "Invalid non-missing values": int(invalid.sum())
        })

invalid_report = pd.DataFrame(
    invalid_checks,
    columns=["Column", "Validation rule", "Invalid non-missing values"]
)

with report_path.open("w", encoding="utf-8") as report:
    report.write("# Task 1: Employee Attrition Data Audit\n\n")

    report.write("## 1. Objective\n\n")
    report.write(
        "Audit the employee attrition dataset before feature engineering "
        "and model development. The audit covers structure, missing values, "
        "duplicates, outliers, categories, invalid measurements, class "
        "imbalance and potential data leakage.\n\n"
    )

    report.write("## 2. Dataset Overview\n\n")
    report.write(f"- Rows: {df.shape[0]}\n")
    report.write(f"- Columns: {df.shape[1]}\n")
    report.write(f"- Numeric columns: {len(df.select_dtypes(include=np.number).columns)}\n")
    report.write(f"- Exact duplicate rows: {df.duplicated().sum()}\n\n")

    report.write("### Column data types\n\n")
    dtype_report = pd.DataFrame({
        "Column": df.columns,
        "Data type": df.dtypes.astype(str).values,
        "Missing count": df.isnull().sum().values
    })
    report.write(dtype_report.to_markdown(index=False))
    report.write("\n\n")

    report.write("## 3. Missing-Value Analysis\n\n")
    report.write(
        f"Columns containing missing values: {len(missing_report)} "
        f"out of {df.shape[1]}.\n\n"
    )
    if missing_report.empty:
        report.write("No missing values were found.\n\n")
    else:
        report.write(missing_report.to_markdown(index=False))
        report.write("\n\n")

    report.write("## 4. Duplicate Record Analysis\n\n")
    report.write(f"- Exact duplicate rows: {df.duplicated().sum()}\n")
    report.write(f"- Duplicate row IDs: {df['row_id'].duplicated().sum()}\n")
    report.write(
        f"- Repeated employee ID occurrences: "
        f"{df['employee_id'].duplicated().sum()}\n"
    )
    report.write(
        f"- Duplicate rows after excluding `row_id`: "
        f"{duplicate_without_row_id}\n\n"
    )
    report.write(
        "The six repeated employee records differ only by `row_id` in the "
        "notebook inspection. Investigate the source and intended meaning "
        "of these records before deciding whether to remove them.\n\n"
    )

    report.write("## 5. Attrition Distribution\n\n")
    attrition_report = pd.DataFrame({
        "Outcome": ["Stayed", "Left"],
        "Records": [
            int((df["left_within_12m"] == 0).sum()),
            int((df["left_within_12m"] == 1).sum())
        ],
        "Percentage": [
            round((df["left_within_12m"] == 0).mean() * 100, 2),
            round((df["left_within_12m"] == 1).mean() * 100, 2)
        ]
    })
    
    report.write(attrition_report.to_markdown(index=False))
    report.write(
        "\n\nThe target is imbalanced. Evaluate future models with "
        "precision, recall, F1-score and suitable threshold-independent "
        "metrics in addition to accuracy.\n\n"
    )
    report.write(
        f"A majority-class baseline that predicts every employee will stay "
        f"would achieve {((df['left_within_12m'] == 0).mean() * 100):.2f}% "
        f"accuracy because {(df['left_within_12m'] == 0).sum():,} out of "
        f"{len(df):,} employees stayed. However, this baseline would identify "
        f"none of the employees who left, giving a recall of 0 for the leaver "
        f"class. Therefore, accuracy alone can be misleading, and recall, "
        f"precision, F1-score and the confusion matrix should also be evaluated.\n\n"
    )


    report.write("## 6. Outlier Detection\n\n")
    report.write(comparison_df.to_markdown(index=False))
    report.write(
        "\n\nIQR and Z-score methods flag potential outliers using different "
        "criteria. A flagged observation is not automatically an error; "
        "investigate plausible extreme values before excluding them.\n\n"
    )

    report.write("## 7. Category Consistency Checks\n\n")
    if category_report.empty:
        report.write(
            "No differences in unique-category counts were detected by "
            "trimming whitespace and ignoring case. This check does not "
            "rule out every possible category inconsistency.\n\n"
        )
    else:
        report.write(
            "The following columns have category values that collapse "
            "to fewer unique values after trimming whitespace and ignoring "
            "case. Review them before normalizing categories.\n\n"
        )
        report.write(category_report.to_markdown(index=False))
        report.write("\n\n")

    report.write("## 8. Invalid Measurement Checks\n\n")
    report.write(
        "These checks flag values outside the stated bounds or negative "
        "values in fields expected to be non-negative. Confirm each rule "
        "against the data dictionary before correcting any record.\n\n"
    )
    report.write(invalid_report.to_markdown(index=False))
    report.write("\n\n")

    report.write("## 9. Data Leakage Review\n\n")
    leakage_table = pd.DataFrame({
        "Column": [
            "row_id", "employee_id", "observation_date",
            "left_within_12m", "voluntary_left_within_12m",
            "exit_date", "resignation_date", "exit_reason",
            "exit_interview_done", "fnf_settlement_amount"
        ],
        "Decision": [
            "Exclude from model inputs",
            "Exclude from model inputs",
            "Use to enforce the prediction-time boundary",
            "Target label; never use as an input feature",
            "Outcome-related label; never use as an input feature",
            "Exclude: exit outcome information",
            "Exclude: resignation outcome information",
            "Exclude: post-outcome information",
            "Exclude: post-outcome information",
            "Exclude: post-outcome information"
        ]
    })
    report.write(leakage_table.to_markdown(index=False))
    report.write(
        "\n\nMonthly attendance and payroll features must not use records "
        "after the relevant observation date. The target is "
        "`left_within_12m`.\n\n"
    )

    report.write("## 10. Data-Quality Decisions\n\n")
    decisions = pd.DataFrame([
        ["Missing numeric values", "Use training-fitted imputation where appropriate."],
        ["Missing categorical values", "Use training-fitted categorical imputation where appropriate."],
        ["Repeated employee IDs", "Investigate record meaning before removing records."],
        ["Potential outliers", "Investigate; do not automatically delete plausible extremes."],
        ["Category inconsistencies", "Review and normalize equivalent values consistently."],
        ["Invalid measurements", "Validate against the data dictionary and document any correction."],
        ["Target and post-outcome fields", "Exclude from predictive input features."],
        ["Employee identifiers", "Exclude from predictive input features."],
        ["Class imbalance", "Use appropriate classification metrics during model evaluation."]
    ], columns=["Issue", "Decision"])
    report.write(decisions.to_markdown(index=False))
    report.write("\n\n")

    
    report.write("## 11. Answer-Key Comparison\n\n")

    report.write(
        "The supplied validation file lists 98 planted data-quality issues "
        "across six categories. The corresponding employee IDs are present "
        "in the source dataset. Existing audit checks identify the affected "
        "issue categories, and all three planted salary anomalies are flagged "
        "by both statistical screening methods.\n\n"
    )

    report.write("### 11.1 What I Missed\n\n")

    report.write(
        "1. **Record-level missing values:** The audit identified 55 missing "
        "values in `commute_distance_km` and 15 in `highest_education`, "
        "matching the answer-key category total. Individual planted records "
        "have not all been independently reconciled.\n"
        "2. **Duplicate rows:** The initial full-row duplicate check returned "
        "zero because the duplicate records have different `row_id` values. "
        "Checking duplicates while excluding `row_id` identified all six "
        "planted duplicate pairs.\n"
        "3. **Salary errors:** The initial outlier analysis detected unusual "
        "salary values, but the three planted errors required employee-level "
        "inspection to identify them specifically.\n"
        "4. **Category inconsistencies:** The initial audit detected "
        "inconsistent `work_location` values but did not initially list every "
        "individual variant.\n\n"
    )

    report.write(
        "**Validation limitation:** Category-level findings and matching "
        "employee IDs do not prove that every planted issue was independently "
        "detected. Full record-level reconciliation remains incomplete; "
        "the audit must not claim all 98 issues were independently detected.\n\n"
    )

    report.write("## 12. Conclusion\n\n")

    report.write(
        "The audit examined dataset structure, missing values, duplicate "
        "candidates, class imbalance, potential outliers, category consistency, "
        "invalid measurements and possible target leakage. It provides a basis "
        "for consistent data cleaning and feature engineering while preserving "
        "the original dataset. The validation-vault comparison documents "
        "category-level findings, with record-level reconciliation still "
        "required before claiming complete verification.\n"
    )

In [ ]:
repeated_employees = df[
    df["employee_id"].duplicated(keep=False)
].sort_values("employee_id")

display(
    repeated_employees[
        [
            "employee_id",
            "observation_date",
            "left_within_12m"
        ]
    ]
)

In [ ]:
# Check whether repeated employee records are identical

repeated_id_counts = df["employee_id"].value_counts()
repeated_id_counts = repeated_id_counts[
    repeated_id_counts > 1
]

for employee_id in repeated_id_counts.index:

    employee_records = df[
        df["employee_id"] == employee_id
    ]

    first_record = employee_records.iloc[0]
    second_record = employee_records.iloc[1]

    different_columns = []

    for column in df.columns:
        if not (
            pd.isna(first_record[column])
            and pd.isna(second_record[column])
        ) and first_record[column] != second_record[column]:

            different_columns.append(column)

    print("\nEmployee ID:", employee_id)
    print("Number of different columns:", len(different_columns))
    print("Different columns:", different_columns)

In [ ]:
# Check duplicates while ignoring the row_id column
duplicate_without_row_id = df.drop(
    columns=["row_id"]
).duplicated().sum()

print(
    "Duplicate rows excluding row_id:",
    duplicate_without_row_id
)

In [ ]:
# Load the data dictionary
dictionary_df = pd.read_csv(DATA_DIR / "data_dictionary.csv")

# Display its column names
print("Data dictionary columns:")
print(dictionary_df.columns.tolist())

# Display the first five rows
display(dictionary_df.head())

In [ ]:
# Select the data dictionary entries for our dataset
attrition_dictionary = dictionary_df[
    dictionary_df["sheet"] == "attrition_dataset"
]

# Display all column definitions
display(
    attrition_dictionary[
        [
            "column",
            "data_type",
            "description",
            "null_count",
            "null_percent"
        ]
    ]
)

In [ ]:
# Find dataset columns missing from the dictionary
dataset_columns = set(df.columns)
dictionary_columns = set(attrition_dictionary["column"])

missing_from_dictionary = dataset_columns - dictionary_columns

print("Dataset columns:", len(dataset_columns))
print("Dictionary entries:", len(dictionary_columns))
print("Columns missing from dictionary:", missing_from_dictionary)

In [ ]:
# Columns that may cause data leakage
leakage_columns = [
    "observation_date",
    "left_within_12m",
    "voluntary_left_within_12m",
    "exit_date",
    "resignation_date",
    "exit_reason",
    "exit_interview_done",
    "fnf_settlement_amount"
]

# Display their definitions from the data dictionary
display(
    attrition_dictionary[
        attrition_dictionary["column"].isin(leakage_columns)
    ][["column", "data_type", "description", "null_percent"]]
)

## 11. Final reproducibility checks and audit decisions

This final section consolidates checks that can be rerun directly from the supplied `attrition_dataset.csv`. It does not modify the source dataset. Validation-vault findings are documented separately in the report; the vault is not required to execute this notebook.


In [ ]:
# Final reproducible audit checks (read-only)
from pathlib import Path

assert not df.empty, "The dataset did not load."
assert "left_within_12m" in df.columns, "Target column is missing."
assert set(df["left_within_12m"].dropna().unique()).issubset({0, 1}), "Unexpected target labels found."

final_checks = pd.DataFrame([
    {"Check": "Dataset rows", "Result": int(df.shape[0]), "Interpretation": "Rows loaded from source CSV"},
    {"Check": "Dataset columns", "Result": int(df.shape[1]), "Interpretation": "Columns loaded from source CSV"},
    {"Check": "Exact duplicate rows", "Result": int(df.duplicated().sum()), "Interpretation": "Exact full-row duplicates"},
    {"Check": "Duplicates excluding row_id", "Result": int(df.drop(columns=["row_id"]).duplicated().sum()), "Interpretation": "Rows equal when generated row_id is ignored"},
    {"Check": "Repeated employee IDs after first occurrence", "Result": int(df["employee_id"].duplicated().sum()), "Interpretation": "Investigate; do not automatically assume all repeats are erroneous"},
    {"Check": "Missing cells", "Result": int(df.isna().sum().sum()), "Interpretation": "All missing cells across all columns"},
    {"Check": "Employees who left", "Result": int((df["left_within_12m"] == 1).sum()), "Interpretation": "Positive target count"},
    {"Check": "Employees who stayed", "Result": int((df["left_within_12m"] == 0).sum()), "Interpretation": "Negative target count"},
    {"Check": "Negative tenure rows", "Result": int((df["tenure_months"] < 0).sum()), "Interpretation": "Impossible tenure candidates"},
    {"Check": "Attendance outside [0, 1]", "Result": int(((df["attendance_rate_3m"] < 0) | (df["attendance_rate_3m"] > 1)).sum()), "Interpretation": "Invalid under the documented rate convention"},
    {"Check": "WFH share outside [0, 1]", "Result": int(((df["wfh_share_3m"] < 0) | (df["wfh_share_3m"] > 1)).sum()), "Interpretation": "Check against the data dictionary before correction"},
])
display(final_checks)

# Leakage fields are documented, not silently dropped from the raw audit table.
post_outcome_columns = [
    "exit_date", "resignation_date", "exit_reason",
    "exit_interview_done", "fnf_settlement_amount",
    "voluntary_left_within_12m"
]
leakage_review = pd.DataFrame({
    "Column": post_outcome_columns,
    "Decision": ["Exclude from model features; event occurs after outcome" if c != "voluntary_left_within_12m" else "Exclude from features; closely reveals target outcome" for c in post_outcome_columns]
})
display(leakage_review)

print("Final checks completed. Source dataframe was not modified.")


## 12. Record-level validation-vault comparison (post-audit)

This section is run only after the independent audit is recorded. The validation file contains both `EMP-` identifiers for the attrition snapshot and `HIST-` identifiers belonging to historical records outside `attrition_dataset.csv`. Do not count the latter as unmatched defects in this dataset; report them as out of scope for this table. The vault is not a model input and should not be committed to the public project.


In [5]:
# Optional post-audit validation comparison. The vault is not required for normal notebook execution.
from pathlib import Path

vault_path = PROJECT_ROOT / "04_Validation_Vault" / "unlock_CX-01_answer_key_planted_dirt.csv"
if not vault_path.exists():
    vault_path = PROJECT_ROOT / "04_Validation_Vault" / "unlock_CX-01_answer_key_planted_dirt.csv"

if vault_path.exists():
    vault = pd.read_csv(vault_path)
    if not {"employee_id", "issue_type", "detail"}.issubset(vault.columns):
        raise ValueError("Validation file is missing expected columns.")

    # The attrition snapshot contains EMP- IDs; HIST- IDs refer to other historical data.
    vault_emp = vault[vault["employee_id"].astype(str).str.startswith("EMP-")].copy()
    vault_hist = vault[vault["employee_id"].astype(str).str.startswith("HIST-")].copy()
    dataset_ids = set(df["employee_id"].astype(str))
    vault_emp["matched_to_attrition_dataset"] = vault_emp["employee_id"].isin(dataset_ids)

    validation_summary = (
        vault.groupby("issue_type").size().rename("Vault total").to_frame()
        .join(vault_emp.groupby("issue_type").size().rename("EMP entries in vault"), how="outer")
        .join(vault_emp[vault_emp["matched_to_attrition_dataset"]].groupby("issue_type").size().rename("EMP entries matched to dataset"), how="outer")
        .fillna(0).astype(int).reset_index()
    )
    display(validation_summary)
    print(f"Vault entries: {len(vault)}; EMP entries: {len(vault_emp)}; HIST entries outside this attrition snapshot: {len(vault_hist)}.")
    print(f"EMP validation entries matched to an employee_id in this dataset: {int(vault_emp['matched_to_attrition_dataset'].sum())} / {len(vault_emp)}.")
    print("A matching ID is not by itself proof that the issue was detected independently; compare the issue detail with the audit checks before marking it found.")
else:
    print("Validation vault not found. Independent audit still runs; copy the unlocked vault to a local validation/ folder only for post-audit comparison.")


Validation vault not found. Independent audit still runs; copy the unlocked vault to a local validation/ folder only for post-audit comparison.


## 12. Validation-vault reconciliation (review only)

This section is an explicit comparison against the supplied CX-01 planted-issue file. It is intended for the review copy. Keep the validation vault out of the public repository unless the programme explicitly requires it. The validation file is a reference list, so matches here show whether planted records exist and what issue types are listed; they do **not** prove the initial audit independently detected every issue.

The cell searches common locations for `unlock_CX-01_answer_key_planted_dirt.csv`. If the file is not present, the notebook reports that and the rest of the audit remains runnable.


In [6]:
from pathlib import Path
import pandas as pd

# Find the source dataset from common Docker, repository, or notebook locations.
possible_roots = [PROJECT_ROOT]
def first_existing(paths):
    for path in paths:
        if path.exists():
            return path.resolve()
    return None

source_path = first_existing([
    root / 'data' / 'attrition_dataset.csv' for root in possible_roots
])
if source_path is None:
    source_path = first_existing([root / 'project' / 'data' / 'attrition_dataset.csv' for root in possible_roots])
if source_path is None:
    raise FileNotFoundError('Could not locate data/attrition_dataset.csv from the notebook working directory.')

validation_path = first_existing([
    root / '04_Validation_Vault' / 'unlock_CX-01_answer_key_planted_dirt.csv' for root in possible_roots
] + [
    root / 'data' / 'validation' / 'unlock_CX-01_answer_key_planted_dirt.csv' for root in possible_roots
] + [
    root / 'Validation Valut' / 'unlock_CX-01_answer_key_planted_dirt.csv' for root in possible_roots
])

source_for_check = pd.read_csv(source_path)
print('Source dataset:', source_path)
print('Source shape:', source_for_check.shape)
if validation_path is None:
    print('Validation vault not found. Copy the unlocked CX-01 CSV into a local validation/ folder to run the comparison.')
    validation_comparison = None
else:
    planted = pd.read_csv(validation_path)
    required = {'employee_id', 'issue_type', 'detail'}
    missing_cols = required - set(planted.columns)
    if missing_cols:
        raise ValueError(f'Validation file is missing columns: {sorted(missing_cols)}')
    planted['employee_id'] = planted['employee_id'].astype(str).str.strip()
    source_for_check['employee_id'] = source_for_check['employee_id'].astype(str).str.strip()
    source_ids = set(source_for_check['employee_id'])
    planted['employee_id_present_in_source'] = planted['employee_id'].isin(source_ids)
    validation_comparison = (planted.groupby('issue_type', dropna=False)
        .agg(planted_cases=('employee_id','size'), unique_employee_ids=('employee_id','nunique'),
             matched_to_source=('employee_id_present_in_source','sum'))
        .reset_index().sort_values('issue_type'))
    display(validation_comparison)
    print('Total planted issue entries:', len(planted))
    print('Entries whose employee_id appears in source:', int(planted['employee_id_present_in_source'].sum()))
    print('Unique employee IDs in planted list:', planted['employee_id'].nunique())
    print('Important: this is a reference-key reconciliation, not proof of independent detection by the audit.')


Source dataset: /workspace/data/attrition_dataset.csv
Source shape: (2389, 42)
Validation vault not found. Copy the unlocked CX-01 CSV into a local validation/ folder to run the comparison.


## 13. Final audit limitations and interpretation

- Keep the original CSV unchanged; this notebook is an audit, not a destructive cleaning script.
- A row flagged by IQR or a Z-score is a statistical outlier candidate, not automatically an error.
- Treat `left_within_12m` as the target. Do not use post-outcome fields such as `exit_date`, `resignation_date`, `exit_reason`, `exit_interview_done`, `fnf_settlement_amount`, or the closely target-revealing `voluntary_left_within_12m` as predictors.
- `observation_date` defines the snapshot boundary and should not be used as a normal predictor without a documented reason.
- The answer-key comparison must be reviewed and reported separately from the initial audit findings.


In [7]:
import json
from pathlib import Path

notebook_path = Path.cwd() / "01_data_audit.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for cell_no, cell in enumerate(notebook["cells"], start=1):
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])

        if any(x in source for x in [
            "to_csv", "savefig", "to_excel",
            "write_text", "open(", "/workspace/reports"
        ]):
            print(f"\n--- Cell {cell_no} ---")
            print(source)


--- Cell 11 ---
# Check the distribution of the target column
print("Target column distribution:")
print(df["left_within_12m"].value_counts())

print("\nTarget distribution in percentage:")
print((df["left_within_12m"].value_counts(normalize=True) * 100).round(2))

# Visualize the target distribution
sns.countplot(data=df, x="left_within_12m")
plt.title("Distribution of Employee Attrition")
plt.xlabel("Left Within 12 Months (0 = No, 1 = Yes)")
plt.ylabel("Number of Records")
plt.savefig(
    REPORT_DIR / "attrition_distribution.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

--- Cell 43 ---
# Select columns that contain missing values
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(
    ascending=False
)

# Plot the 10 columns with the most missing values
if not missing_counts.empty:
    missing_counts.head(10).plot(kind="bar")

    plt.title("Top 10 Columns with Missing Values")
    plt.xlabel("Column")
    plt.ylabel("Numbe